# 22 Reading JSON and Nested Data

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Read single-line (JSON Lines) and multi-line JSON, work with nested structs and arrays, flatten them with dot notation and <code>explode</code>, and parse JSON strings inside columns with <code>from_json</code>, <code>get_json_object</code> and <code>schema_of_json</code>.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
APIs, event streams, logs and NoSQL exports all produce JSON, and it is almost always <b>nested</b>: an order contains a customer object and an array of line items. Turning that into clean tables (one row per order, one row per line item) is a daily Bronze-to-Silver task, and <code>explode</code> vs <code>explode_outer</code> is a favourite interview question.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 JSON in Spark</b>
<ul><li><b>JSON Lines</b> (one JSON object per line) is the default and is splittable, so large files are read in parallel</li><li><b>Multi-line JSON</b> (a pretty-printed object or one big array) needs <code>multiLine=True</code>, and each file is read by one task</li><li>JSON objects become <code>STRUCT</code> columns, JSON arrays become <code>ARRAY</code> columns</li><li>Flatten structs with <code>col("a.b")</code> or <code>select("a.*")</code></li><li>Turn arrays into rows with <code>explode</code> (drops empty/null arrays) or <code>explode_outer</code> (keeps them as null)</li><li>JSON stored as a <b>string</b> inside a column is parsed with <code>from_json(col, schema)</code></li></ul>
</div>

```
{"order_id": 1, "customer": {"id": "C1", "city": "Chennai"}, "items": [{"sku": "A", "qty": 2}, {"sku": "B", "qty": 1}]}

order_id | customer.id | customer.city | items                    explode(items) ->  order_id | sku | qty
1        | C1          | Chennai       | [{A,2},{B,1}]                                 1        | A   | 2
                                                                                       1        | B   | 1
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
An e-commerce platform publishes order events from its checkout API. Each event has an order header, a nested customer object, a nested shipping address, an array of line items, and a free-form <code>metadata</code> field that is itself a JSON string. The Silver layer produces <code>orders</code> (one row per order with flattened customer and address fields) and <code>order_items</code> (one row per line item, via <code>explode</code>), and parses <code>metadata</code> with <code>from_json</code>.
</div>

## Syntax

```python
from pyspark.sql import functions as F

df = spark.read.json(path)                                   # JSON Lines
df = spark.read.option("multiLine", True).json(path)          # pretty-printed / array files
df = spark.read.schema(ddl).json(path)                        # explicit schema

df.select("order_id", "customer.city", "customer.*")          # flatten structs
df.select("order_id", F.explode("items").alias("item"))       # one row per array element
df.select(F.from_json("payload", "a INT, b STRING").alias("p"))  # parse a JSON string column
df.select(F.get_json_object("payload", "$.a"))                # extract one value by path
```

## Setup: volume and sample JSON files

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates the lesson folder and writes a JSON Lines file of orders, a multi-line JSON file, and a file with a malformed line.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The <code>BASE</code> path, then the list of created files.
</div>

In [0]:
CATALOG, SCHEMA, VOLUME = "workspace", "spark_basics", "raw_files"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/lesson22"
print("BASE =", BASE)

In [0]:
import os, json

orders_jsonl = [
    {"order_id": 1, "order_ts": "2024-09-01T10:15:00", "customer": {"id": "C1", "name": "Amal", "address": {"city": "Chennai", "country": "IN"}},
     "items": [{"sku": "LAPTOP", "qty": 1, "price": 899.0}, {"sku": "MOUSE", "qty": 2, "price": 15.5}],
     "tags": ["prime", "gift"], "metadata": '{"channel": "app", "coupon": "SAVE10"}'},
    {"order_id": 2, "order_ts": "2024-09-01T11:00:00", "customer": {"id": "C2", "name": "John", "address": {"city": "London", "country": "UK"}},
     "items": [{"sku": "DESK", "qty": 1, "price": 250.0}],
     "tags": [], "metadata": '{"channel": "web"}'},
    {"order_id": 3, "order_ts": "2024-09-02T09:30:00", "customer": {"id": "C3", "name": "Vivek", "address": None},
     "items": None, "metadata": None},
]

def write(rel, text):
    path = f"{BASE}/{rel}"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

write("orders/orders.jsonl", "\n".join(json.dumps(o) for o in orders_jsonl) + "\n")

write("multiline/stores.json", json.dumps(
    [{"store_id": "S1", "city": "Chennai", "open": True}, {"store_id": "S2", "city": "Dubai", "open": False}],
    indent=2,
))

write("dirty/events.jsonl", '{"id": 1, "event": "click"}\n{"id": 2, "event": "view"\n{"id": "three", "event": "buy"}\n')

for root, _, names in os.walk(BASE):
    for n in sorted(names):
        print(os.path.join(root, n).replace(BASE, "BASE"))

## 1. Read JSON Lines and look at the inferred schema

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads the orders file and prints the nested schema Spark infers.<br><br>
<b>Why it matters</b><br>JSON inference discovers structs and arrays automatically. Note that the order of fields is <b>alphabetical</b>, integers become <code>long</code>, and a field missing from some records is simply null there.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A schema tree with <code>customer</code> as a struct (containing an <code>address</code> struct), <code>items</code> as an array of structs, <code>tags</code> as an array of strings, and <code>metadata</code> as a string. 3 rows.
</div>

In [0]:
orders = spark.read.json(f"{BASE}/orders/")
orders.printSchema()
orders.show(truncate=False)

## 2. Flatten structs

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Selects nested fields with dot notation and expands a whole struct with <code>.*</code>.<br><br>
<b>Why it matters</b><br>Most consumers (BI tools, SQL users) want flat columns. Dot notation reaches into structs at any depth.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
One row per order with <code>customer_id</code>, <code>customer_name</code>, <code>city</code> and <code>country</code>. Vivek's city and country are null, because his address is null.
</div>

In [0]:
from pyspark.sql import functions as F

flat = orders.select(
    "order_id",
    F.to_timestamp("order_ts").alias("order_ts"),
    F.col("customer.id").alias("customer_id"),
    F.col("customer.name").alias("customer_name"),
    F.col("customer.address.city").alias("city"),
    F.col("customer.address.country").alias("country"),
)
flat.show()

# ".*" expands every field of a struct into its own column.
orders.select("order_id", "customer.address.*").show()

## 3. Explode arrays into rows: `explode` vs `explode_outer`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates one row per line item with <code>explode</code>, then repeats with <code>explode_outer</code>.<br><br>
<b>Why it matters</b><br><code>explode</code> <b>drops</b> rows whose array is null or empty, which silently loses orders. <code>explode_outer</code> keeps them with a null element. Choose deliberately.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>explode</code>: 3 item rows (orders 1 and 2 only). <code>explode_outer</code>: 4 rows, including order 3 with null item fields.
</div>

In [0]:
items = orders.select("order_id", F.explode("items").alias("item"))
print("explode rows:", items.count())
items.select("order_id", "item.sku", "item.qty", "item.price").show()

items_outer = orders.select("order_id", F.explode_outer("items").alias("item"))
print("explode_outer rows:", items_outer.count())
items_outer.select("order_id", "item.*").show()

## 4. Position, totals, and array functions without exploding

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Uses <code>posexplode</code> to keep the line number, then computes order totals two ways: explode + groupBy, and higher-order functions that work directly on arrays.<br><br>
<b>Why it matters</b><br>Exploding multiplies rows, so it costs memory and often a shuffle to re-aggregate. Array functions like <code>size</code>, <code>array_contains</code>, <code>transform</code>, <code>filter</code> and <code>aggregate</code> compute per-row results <b>without</b> exploding.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Line numbers 0 and 1 for order 1. Both total calculations give 930.0 for order 1 and 250.0 for order 2. <code>has_gift</code> is true only for order 1.
</div>

In [0]:
orders.select("order_id", F.posexplode("items").alias("line_no", "item")) \
      .select("order_id", "line_no", "item.sku").show()

# Way 1: explode, then aggregate back
(
    orders.select("order_id", F.explode("items").alias("i"))
    .groupBy("order_id")
    .agg(F.sum(F.col("i.qty") * F.col("i.price")).alias("total_via_explode"))
    .orderBy("order_id")
    .show()
)

# Way 2: higher-order functions on the array itself (no explode, no shuffle)
orders.select(
    "order_id",
    F.size("items").alias("n_items"),
    F.aggregate("items", F.lit(0.0), lambda acc, i: acc + i["qty"] * i["price"]).alias("total_via_aggregate"),
    F.transform("items", lambda i: i["sku"]).alias("skus"),
    F.filter("items", lambda i: i["price"] > 100).alias("expensive_items"),
    F.array_contains("tags", "gift").alias("has_gift"),
).show(truncate=False)

## 5. JSON strings inside a column: `from_json`, `get_json_object`, `schema_of_json`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Parses the <code>metadata</code> column, which holds JSON <b>as text</b>, with three different tools.<br><br>
<b>Why it matters</b><br>Kafka messages, database exports and API payloads often store JSON as a string column. <code>get_json_object</code> pulls single values quickly. <code>from_json</code> parses the whole thing into a typed struct. <code>schema_of_json</code> helps you write the schema from a sample.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>channel</code> is <code>app</code>, <code>web</code> and null. <code>coupon</code> is <code>SAVE10</code> for order 1 and null otherwise. <code>schema_of_json</code> prints a DDL string such as <code>STRUCT&lt;channel: STRING, coupon: STRING&gt;</code>.
</div>

In [0]:
# Quick extraction of single values with a JSON path
orders.select("order_id", F.get_json_object("metadata", "$.channel").alias("channel")).show()

# Infer a schema from one example value
sample = orders.filter("order_id = 1").select("metadata").first()[0]
print(spark.range(1).select(F.schema_of_json(F.lit(sample))).first()[0])

# Parse into a typed struct, then flatten
meta_schema = "channel STRING, coupon STRING"
orders.select("order_id", F.from_json("metadata", meta_schema).alias("meta")) \
      .select("order_id", "meta.*").show()

## 6. Multi-line JSON files

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads a pretty-printed JSON array, first without and then with <code>multiLine=True</code>.<br><br>
<b>Why it matters</b><br>Without <code>multiLine</code>, Spark expects one object per line, so a pretty-printed file becomes corrupt records. With <code>multiLine</code>, a top-level array becomes one row per element.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Without <code>multiLine</code>: a schema with only <code>_corrupt_record</code>. With <code>multiLine</code>: 2 stores with <code>store_id</code>, <code>city</code> and <code>open</code>.
</div>

In [0]:
spark.read.json(f"{BASE}/multiline/").printSchema()

stores = spark.read.option("multiLine", True).json(f"{BASE}/multiline/")
stores.show()

## 7. Explicit schemas and bad JSON records

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads a file with a broken line and a wrongly typed value, in <code>PERMISSIVE</code> mode with a corrupt-record column and an explicit schema.<br><br>
<b>Why it matters</b><br>Same principles as CSV: an explicit schema is faster and stricter, and malformed lines are kept for investigation instead of crashing the job.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
3 rows: event 1 is clean, the broken line has only <code>_corrupt_record</code>, and the row with <code>"id": "three"</code> has a null <code>id</code> plus its raw text in <code>_corrupt_record</code>.
</div>

In [0]:
events = (
    spark.read.schema("id INT, event STRING, _corrupt_record STRING")
    .option("mode", "PERMISSIVE")
    .json(f"{BASE}/dirty/")
)
events.show(truncate=False)

## 8. Build the Silver tables: orders and order_items

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Produces two clean, flat DataFrames from the nested source: one row per order and one row per item.<br><br>
<b>Why it matters</b><br>This is the standard normalization step for nested data. The header table keeps one row per order, and the child table repeats the key (<code>order_id</code>) so the two can be joined back.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>silver_orders</code>: 3 rows with flat columns. <code>silver_order_items</code>: 3 rows with <code>order_id</code>, <code>line_no</code>, <code>sku</code>, <code>qty</code>, <code>price</code>, <code>line_total</code>.
</div>

In [0]:
silver_orders = orders.select(
    "order_id",
    F.to_timestamp("order_ts").alias("order_ts"),
    F.col("customer.id").alias("customer_id"),
    F.col("customer.address.city").alias("city"),
    F.col("customer.address.country").alias("country"),
    F.from_json("metadata", meta_schema)["channel"].alias("channel"),
    F.coalesce(F.size("items"), F.lit(0)).alias("n_items"),
)

silver_order_items = (
    orders.select("order_id", F.posexplode("items").alias("line_no", "i"))
    .select("order_id", "line_no", "i.sku", "i.qty", "i.price",
            (F.col("i.qty") * F.col("i.price")).alias("line_total"))
)
silver_orders.show()
silver_order_items.show()

## Under the hood

```
JSON Lines (default)          multiLine=True
  file split into chunks        one task per FILE (not splittable)
  each line parsed on its own   whole file parsed as one document

Inference: an extra pass over (a sample of) the data. Set samplingRatio < 1.0 to infer from fewer rows.

explode(items)  -> Generate node: each input row produces N output rows (narrow, but multiplies data)
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> `explode` and struct flattening are row-level, with no shuffle. But re-aggregating after an explode needs a `groupBy`, which **does** shuffle.

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> reading with an explicit schema is lazy. Inference runs a job immediately.

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A <code>Generate explode(items...)</code> node. Below it, the optimizer has added <code>Filter (size(items) &gt; 0 AND isnotnull(items))</code>, because <code>explode</code> can never produce rows from empty or null arrays, so those rows are removed as early as possible. The scan reads only <code>items</code> and <code>order_id</code> (column pruning).
</div>

In [0]:
orders.select("order_id", F.explode("items").alias("item")).explain()

In [0]:
# Optional cleanup: remove this lesson's files.
dbutils.fs.rm(BASE, True)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: the DataFrame has only a <code>_corrupt_record</code> column</b><br>
The file is pretty-printed or a JSON array. Add <code>multiLine=True</code>.<br><br>
<b>⛔ Problem: orders disappear after explode</b><br><code>explode</code> drops null and empty arrays. Use <code>explode_outer</code>.<br><br>
<b>⛔ Problem: row counts double after a second explode</b><br>Exploding two arrays one after another makes a cross product of their elements. Explode one array per child table, or zip them with <code>arrays_zip</code> first.<br><br>
**⛔ Problem: <code>from_json</code> returns all nulls**<br>The schema doesn't match the JSON (field names are case sensitive, or wrong types). Check with <code>schema_of_json</code> on a sample.<br><br>
<b>⛔ Problem: inference is slow on large JSON</b><br>Provide a schema, or set <code>samplingRatio</code>.<br><br>
<b>⛔ Problem: field names with dots or spaces</b><br>Use backticks: <code>F.col("`user.name`")</code>.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How do you read multi-line JSON in Spark?</b><br>
With <code>option("multiLine", True)</code>. By default Spark expects JSON Lines, one object per line. Multi-line files can't be split, so each file is processed by one task.<br><br>

<b>🎤 2. What is the difference between <code>explode</code> and <code>explode_outer</code>?</b><br>
Both create one row per array element. <code>explode</code> drops rows where the array is null or empty, while <code>explode_outer</code> keeps them with a null element.<br><br>

<b>🎤 3. How do you flatten a nested JSON structure?</b><br>
Use dot notation or <code>struct.*</code> for nested objects, and <code>explode</code> or <code>posexplode</code> for arrays, usually producing separate tables for parent and child records linked by a key.<br><br>

<b>🎤 4. How do you parse a JSON string stored in a column?</b><br>
<code>from_json(col, schema)</code> returns a typed struct. <code>get_json_object(col, '$.path')</code> extracts a single value as a string. <code>schema_of_json</code> helps derive the schema from a sample.<br><br>

<b>🎤 5. How can you work with arrays without exploding them?</b><br>
Higher-order functions such as <code>transform</code>, <code>filter</code>, <code>exists</code>, <code>aggregate</code> and <code>array_contains</code> operate on the array inside each row, which avoids multiplying rows and re-aggregating.<br><br>

<b>🎤 6. Why provide a schema when reading JSON?</b><br>
Inference needs an extra pass, can mis-type fields (for example <code>long</code> vs <code>decimal</code>), and changes when the data changes. A schema is faster and makes the contract explicit.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A column <code>items</code> is an array of structs. Some rows have an empty array. Which function returns one row per element while keeping the rows with empty arrays?</b><br>
A. <code>explode(items)</code><br>
B. <code>explode_outer(items)</code><br>
C. <code>flatten(items)</code><br>
D. <code>posexplode(items)</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>explode</code> and <code>posexplode</code> drop null or empty arrays. <code>flatten</code> merges nested arrays into one array, it doesn't create rows.</details><br><br>

<b>Q2. A column <code>payload</code> contains JSON text such as <code>{"device": {"id": 7}}</code>. Which Databricks SQL syntax extracts the device id?</b><br>
A. <code>payload.device.id</code><br>
B. <code>payload:device.id</code><br>
C. <code>payload-&gt;device-&gt;id</code><br>
D. <code>payload['device']['id']</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> Databricks SQL supports the <code>:</code> operator to extract fields from JSON strings. Dot notation works on struct columns, not on strings.</details><br><br>

<b>Q3. Which option is required to read a JSON file containing a single pretty-printed array of objects?</b><br>
A. <code>mode = 'PERMISSIVE'</code><br>
B. <code>multiLine = true</code><br>
C. <code>inferSchema = true</code><br>
D. <code>lineSep = '\n'</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Write a JSON Lines file of 3 users, each with <code>user_id</code>, a <code>profile</code> struct (<code>name</code>, <code>age</code>), and an array <code>devices</code> of structs (<code>type</code>, <code>os</code>). Give one user an empty <code>devices</code> array</li><li>Read it with an <b>explicit</b> DDL schema</li><li>Produce <code>users</code> (flat: <code>user_id</code>, <code>name</code>, <code>age</code>, <code>n_devices</code>) without exploding</li><li>Produce <code>user_devices</code> (one row per device), keeping the user with no devices</li><li>Add a column <code>has_ios</code> using a higher-order function</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Write the file
users_data = [
    {"user_id": 1, "profile": {"name": "Amal", "age": 34}, "devices": [{"type": "phone", "os": "iOS"}, {"type": "laptop", "os": "Windows"}]},
    {"user_id": 2, "profile": {"name": "Sara", "age": 29}, "devices": [{"type": "phone", "os": "Android"}]},
    {"user_id": 3, "profile": {"name": "John", "age": 41}, "devices": []},
]
write("practice/users.jsonl", "\n".join(json.dumps(u) for u in users_data) + "\n")

# 2. Explicit schema
user_schema = "user_id INT, profile STRUCT<name: STRING, age: INT>, devices ARRAY<STRUCT<type: STRING, os: STRING>>"
users_raw = spark.read.schema(user_schema).json(f"{BASE}/practice/")

# 3. Flat users without exploding, 5. higher-order function
users = users_raw.select(
    "user_id", "profile.name", "profile.age",
    F.size("devices").alias("n_devices"),
    F.exists("devices", lambda d: d["os"] == "iOS").alias("has_ios"),
)
users.show()

# 4. One row per device, keeping users without devices
users_raw.select("user_id", F.explode_outer("devices").alias("d")).select("user_id", "d.type", "d.os").show()

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>JSON Lines is the default and splittable. Pretty-printed or array files need <code>multiLine=True</code></li><li>Objects become <code>STRUCT</code>, arrays become <code>ARRAY</code>. Inferred fields are sorted alphabetically</li><li>Flatten structs with <code>a.b</code> or <code>a.*</code>. Turn arrays into rows with <code>explode</code> / <code>explode_outer</code> / <code>posexplode</code></li><li><code>explode</code> drops null and empty arrays. <code>explode_outer</code> keeps them</li><li>Use higher-order functions (<code>transform</code>, <code>filter</code>, <code>exists</code>, <code>aggregate</code>) to avoid exploding</li><li>JSON in string columns: <code>from_json</code> (typed), <code>get_json_object</code> (one value), <code>schema_of_json</code> (help)</li><li>Normalize nested data into parent and child tables linked by a key</li></ul>
</div>

| Task | Code |
|---|---|
| Read | `spark.read.schema(ddl).json(path)` |
| Multi-line | `.option("multiLine", True)` |
| Struct field | `F.col("customer.address.city")`, `select("customer.*")` |
| Array to rows | `F.explode_outer("items")`, `F.posexplode("items")` |
| Parse string | `F.from_json("payload", ddl)` |
| One value | `F.get_json_object("payload", "$.a.b")` |
| Array maths | `F.aggregate("items", F.lit(0.0), lambda acc, x: acc + x["qty"])` |
| SQL (Databricks) | `payload:device.id` |

## Git commit

```
git add 01_spark_basics/22_reading_json.ipynb
git commit -m "add 22 reading json notebook"
```